# Stage C — Geographic analysis
## Where the literature comes from, and what it is about

**AI adoption in urban planning governance: a systematic review**
Lartey & Law (2025), *Landscape and Urban Planning* 258, 105337

---

Section 3.2.2 and 4.1. The paper reports **588 studies across 83 countries**,
concentrated in Europe, North America and Asia, with Africa, Central America and
Oceania markedly lower.

This is the notebook where methodological care matters most, because the headline
finding is a claim about *absence*, and absence is exactly what a geocoder produces
when it fails. Three things follow from that.

### 1. Coverage is the first thing to check

A country missing from your reference table cannot be found, and a country that is
never found reads as a country that never publishes. `data/reference/countries.csv`
therefore covers **all 54 African countries and all 7 Central American ones** —
precisely the regions the finding concerns. Stage C1 prints the coverage by region
so you can confirm it before trusting any result.

### 2. "About" and "from" are different claims

- **Study countries** — places the work is *about*, from title, abstract and keywords.
- **Author countries** — where the authors are *based*, from affiliations.

Both are extracted, separately. A review can be *about* Nairobi and written entirely
in Europe; collapsing those into one number turns a claim about research capacity
into a claim about research attention, or the reverse.

### 3. Journal names are not evidence of location

*Journal of the American Planning Association* would tag every paper it publishes as
United States regardless of where the study was done. Journal is used for discipline
in notebook 02 and deliberately not used here.

**Reads** `data/corpus.csv` (or `corpus_classified.csv` if notebook 02 has run)
**Writes** `outputs/figures/figure2_geographic.png` and country/region tables.

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def _repo_relative(p):
    """Resolve whether the kernel started in notebooks/ or the repository root."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


CONFIG = {"data_dir": "../data", "output_dir": "../outputs", "seed": 42, "dpi": 200}
SEED = CONFIG["seed"]
np.random.seed(SEED)

DATA_DIR = _repo_relative(CONFIG["data_dir"])
OUT_DIR = _repo_relative(CONFIG["output_dir"])
FIG_DIR, TAB_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CONFIG["dpi"],
                     "savefig.bbox": "tight", "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold"})

PALETTE = ["#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"]

CORPUS_PATH = (DATA_DIR / "corpus_classified.csv"
               if (DATA_DIR / "corpus_classified.csv").exists()
               else DATA_DIR / "corpus.csv")
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"{CORPUS_PATH.resolve()} not found.\n"
        f"Run 01_corpus_assembly.ipynb first - it writes this file.")

corpus = pd.read_csv(CORPUS_PATH)
corpus["Year"] = pd.to_numeric(corpus["Year"], errors="coerce")
corpus = corpus.dropna(subset=["Year"])
corpus["Year"] = corpus["Year"].astype(int)

IS_DEMO = corpus["SourceFile"].astype(str).eq("SYNTHETIC").any() \
    if "SourceFile" in corpus.columns else False


def stamp(fig, demo=None, text="DEMO DATA\nNOT PAPER RESULTS"):
    """Mark synthetic output. `demo` defaults to the corpus, but a figure built from
    another source (e.g. the VOSviewer network) passes its own flag."""
    if (IS_DEMO if demo is None else demo):
        fig.text(0.5, 0.5, text, fontsize=42, color="grey", alpha=0.13,
                 ha="center", va="center", rotation=30, zorder=1000, weight="bold")
    return fig


def finish(fig, name, demo=None, stamp_text="DEMO DATA\nNOT PAPER RESULTS"):
    stamp(fig, demo=demo, text=stamp_text)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    plt.show()
    print(f"  saved: {path}")


print(f"Corpus: {len(corpus):,} records, {corpus['Year'].min()}-{corpus['Year'].max()}")
print(f"Paper for comparison: 588 records across 83 countries")
if IS_DEMO:
    print("\n" + "!" * 66)
    print("This corpus is the SYNTHETIC demo set. Figures will be stamped.")
    print("!" * 66)

---
## C1 — Load the reference table and check coverage

`data/reference/countries.csv` is a plain CSV: country, centroid, region, and a
pipe-separated alias list holding demonyms and major cities. Edit it freely — add a
city your corpus mentions often, or a country name variant your database uses.

Matching is longest-first, so *South Africa* wins over *Africa* and *United Arab
Emirates* over *United*. A small set of bare names is excluded as too ambiguous:
*Georgia* is far more often the US state in this literature, so it is matched only via
*Tbilisi* or *Republic of Georgia*.

In [ ]:
REF_PATH = DATA_DIR / "reference" / "countries.csv"
if not REF_PATH.exists():
    raise FileNotFoundError(f"{REF_PATH.resolve()} not found. It ships with the repo.")

ref = pd.read_csv(REF_PATH)
COUNTRY_META = {r.Country: (r.Latitude, r.Longitude, r.Region) for r in ref.itertuples()}
REGIONS = sorted(ref["Region"].unique())

LOOKUP = {}
for r in ref.itertuples():
    LOOKUP[r.Country.lower()] = r.Country
    if isinstance(r.Aliases, str) and r.Aliases.strip():
        for a in r.Aliases.split("|"):
            if a.strip():
                LOOKUP[a.strip().lower()] = r.Country

# Bare names too ambiguous to match on their own in this literature
AMBIGUOUS_BARE_NAMES = {"georgia"}
_KEYS = [k for k in sorted(LOOKUP, key=len, reverse=True)
         if k not in AMBIGUOUS_BARE_NAMES]
PATTERN = re.compile(r"\b(" + "|".join(re.escape(k) for k in _KEYS) + r")\b",
                     re.IGNORECASE)

coverage = ref.groupby("Region").size().rename("countries")
print(f"Reference table: {len(ref)} countries, {len(LOOKUP):,} lookup keys\n")
display(coverage.to_frame())
print("\nCoverage check for the regions the paper's finding concerns:")
print(f"  Africa:          {coverage.get('Africa', 0)} of 54 countries")
print(f"  Central America: {coverage.get('Central America', 0)} of 7 countries")
print("\n  A gap here would show up as an absence of research, not an absence of")
print("  coverage. Confirm these are complete before reading anything into a")
print("  low regional count.")

---
## C2 — Extract locations

Every country mentioned is captured, not just the first. A paper comparing Accra and
Nairobi is evidence for both; recording only Ghana understates Kenya, and doing that
systematically across a corpus biases exactly the regional totals the paper reports.

Records with no location found are counted and reported rather than dropped, because
the size of that group tells you how much of the corpus the geographic claim actually
rests on.

In [ ]:
def find_countries(text):
    """Every country mentioned, deduplicated, in order of first appearance."""
    if not isinstance(text, str) or not text.strip():
        return []
    seen, out = set(), []
    for m in PATTERN.finditer(text):
        c = LOOKUP[m.group(1).lower()]
        if c not in seen:
            seen.add(c); out.append(c)
    return out


# What the studies are ABOUT - never includes Journal
about_blob = pd.Series("", index=corpus.index)
for f in ("Title", "Abstract", "Keyword"):
    if f in corpus.columns:
        about_blob = about_blob + " " + corpus[f].fillna("").astype(str)
corpus["study_countries"] = about_blob.map(find_countries)

# Where the AUTHORS are based
if "Affiliation" in corpus.columns and corpus["Affiliation"].notna().any():
    corpus["author_countries"] = corpus["Affiliation"].fillna("").astype(str).map(find_countries)
    HAS_AFFIL = corpus["author_countries"].str.len().gt(0).any()
else:
    corpus["author_countries"] = [[] for _ in range(len(corpus))]
    HAS_AFFIL = False

n_loc = int(corpus["study_countries"].str.len().gt(0).sum())
multi = int(corpus["study_countries"].str.len().gt(1).sum())
all_c = {c for lst in corpus["study_countries"] for c in lst}

print(f"Records with a study location : {n_loc:,} of {len(corpus):,} ({n_loc / len(corpus):.0%})")
print(f"  mentioning more than one     : {multi:,}")
print(f"  distinct countries found     : {len(all_c)}   (paper: 83)")
if HAS_AFFIL:
    na = int(corpus["author_countries"].str.len().gt(0).sum())
    print(f"Records with an author country: {na:,} ({na / len(corpus):.0%})")
else:
    print("\nNo usable affiliations - author-based analysis is skipped below.")
    print("Re-export with affiliations if you need the 'research capacity' reading")
    print("as well as the 'research attention' one.")

if len(corpus) - n_loc:
    print(f"\n{len(corpus) - n_loc:,} record(s) have no detectable location. Regional")
    print("totals below describe only the located subset - state that when citing them.")

In [ ]:
def counts_by_country(col):
    tally = Counter()
    for lst in corpus[col]:
        tally.update(lst)
    if not tally:
        return pd.DataFrame(columns=["Country", "Records", "Region", "Latitude", "Longitude"])
    rows = [{"Country": c, "Records": n, "Region": COUNTRY_META[c][2],
             "Latitude": COUNTRY_META[c][0], "Longitude": COUNTRY_META[c][1]}
            for c, n in tally.items()]
    return pd.DataFrame(rows).sort_values("Records", ascending=False).reset_index(drop=True)


def counts_by_region(col):
    tally = Counter()
    for lst in corpus[col]:
        tally.update({COUNTRY_META[c][2] for c in lst})
    s = pd.Series({r: tally.get(r, 0) for r in REGIONS}, name="Records")
    s.index.name = "Region"
    return s.sort_values(ascending=False)


country_counts = counts_by_country("study_countries")
region_counts = counts_by_region("study_countries")

# Published regional totals (Figure 2), for comparison only
PAPER_REGIONS = {"Europe": 187, "Asia": 149, "North America": 143, "South America": 40,
                 "Africa": 35, "Oceania": 19, "Central America": 15}

cmp_r = pd.DataFrame({"yours": region_counts,
                      "paper": pd.Series(PAPER_REGIONS)}).fillna(0).astype(int)
cmp_r["yours_%"] = (cmp_r["yours"] / cmp_r["yours"].sum() * 100).round(1)
cmp_r["paper_%"] = (cmp_r["paper"] / cmp_r["paper"].sum() * 100).round(1)

print("Regional distribution vs the published Figure 2")
display(cmp_r.sort_values("paper", ascending=False))
print("\nTop 15 countries:")
display(country_counts.head(15))

country_counts.to_csv(TAB_DIR / "country_counts.csv", index=False)
cmp_r.to_csv(TAB_DIR / "region_counts.csv")
if HAS_AFFIL:
    counts_by_country("author_countries").to_csv(TAB_DIR / "author_country_counts.csv",
                                                 index=False)

---
## C3 — Figure 2: the world map and regional theme mix

The paper's Figure 2 pairs a choropleth of study counts with a table of keyword
frequency by region. Both panels are rebuilt from the corpus below.

The map uses country centroids rather than polygon boundaries — no geopandas
dependency, no shapefile download, and the point is relative volume rather than exact
borders. If you want a true choropleth, `geopandas` plus Natural Earth will slot in
where the scatter is drawn.

In [ ]:
EXTRACTION_KEYWORDS = {
    "Decision making":        ["decision-making", "decision making"],
    "Urban Planning":         ["urban planning", "city planning"],
    "Artificial Intelligence": ["artificial intelligence", " ai ", "machine learning"],
    "Policy making":          ["policy making", "policymaking", "policy formulation"],
}

blob = pd.Series("", index=corpus.index)
for f in ("Title", "Abstract", "Keyword"):
    if f in corpus.columns:
        blob = blob + " " + corpus[f].fillna("").astype(str)
blob = blob.str.lower()
for k, terms in EXTRACTION_KEYWORDS.items():
    corpus[f"kw_{k}"] = blob.str.contains(
        "|".join(re.escape(t) for t in terms), regex=True).astype(int)

# Region x keyword: share of each region's records matching each keyword
rows = {}
for region in REGIONS:
    mask = corpus["study_countries"].map(
        lambda lst: any(COUNTRY_META[c][2] == region for c in lst))
    if mask.sum() == 0:
        continue
    sub = corpus[mask]
    rows[region] = {k: sub[f"kw_{k}"].sum() for k in EXTRACTION_KEYWORDS}
region_theme = pd.DataFrame(rows).T.reindex(columns=list(EXTRACTION_KEYWORDS)).fillna(0)
region_theme_pct = (region_theme.div(region_theme.sum(axis=0), axis=1) * 100).round(0)

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 1, height_ratios=[2.2, 1], hspace=0.18)

ax = fig.add_subplot(gs[0])
if len(country_counts):
    sizes = np.clip(country_counts["Records"] / country_counts["Records"].max() * 780, 22, 800)
    sc = ax.scatter(country_counts["Longitude"], country_counts["Latitude"],
                    s=sizes, c=country_counts["Records"], cmap="YlGnBu",
                    alpha=0.82, edgecolor="#37474F", linewidth=0.6, zorder=3)
    plt.colorbar(sc, ax=ax, label="Records", fraction=0.022, pad=0.01)
    for _, r in country_counts.head(14).iterrows():
        ax.annotate(r["Country"], (r["Longitude"], r["Latitude"]), fontsize=7,
                    xytext=(0, 11), textcoords="offset points", ha="center")
ax.axhline(0, color="grey", lw=0.4, ls=":")
ax.set(xlim=(-180, 180), ylim=(-60, 84), xlabel="Longitude", ylabel="Latitude",
       title=f"Geographic distribution — {n_loc:,} located records, "
             f"{len(all_c)} countries")

ax = fig.add_subplot(gs[1])
sns.heatmap(region_theme_pct.T, annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "% of keyword's records"}, linewidths=0.5, ax=ax)
ax.set(title="Keyword frequency by region (% of each keyword's located records)",
       xlabel="", ylabel="")
for i, region in enumerate(region_theme_pct.index):
    ax.text(i + 0.5, -0.18, f"n={int(region_counts.get(region, 0))}",
            ha="center", fontsize=7.5, transform=ax.get_xaxis_transform())

fig.suptitle("Figure 2 — Geographic distribution and regional keyword frequencies",
             fontsize=13, weight="bold")
finish(fig, "figure2_geographic")
region_theme_pct.to_csv(TAB_DIR / "region_keyword_pct.csv")

---
## C4 — About vs from

If your corpus carries affiliations, this is the comparison that separates *research
attention* from *research capacity*. A region can score respectably on study countries
while barely appearing among author countries — which is a finding about who studies
whom, and a stronger claim than either column alone.

Skipped automatically when no affiliations are present.

In [ ]:
if not HAS_AFFIL:
    print("No affiliation data in this corpus - nothing to compare.")
    print("Re-export from your database with affiliations included and re-run.")
else:
    about = counts_by_region("study_countries")
    frm = counts_by_region("author_countries")
    cmp_af = pd.DataFrame({"studied (about)": about, "authored (from)": frm}).fillna(0).astype(int)
    cmp_af["about_%"] = (cmp_af["studied (about)"] / cmp_af["studied (about)"].sum() * 100).round(1)
    cmp_af["from_%"] = (cmp_af["authored (from)"] / max(cmp_af["authored (from)"].sum(), 1) * 100).round(1)
    cmp_af["gap"] = (cmp_af["about_%"] - cmp_af["from_%"]).round(1)
    display(cmp_af.sort_values("about_%", ascending=False))

    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(len(cmp_af))
    ax.bar(x - 0.2, cmp_af["about_%"], 0.4, label="Studied (about)", color=PALETTE[1])
    ax.bar(x + 0.2, cmp_af["from_%"], 0.4, label="Authored (from)", color=PALETTE[3])
    ax.set_xticks(x); ax.set_xticklabels(cmp_af.index, rotation=30, ha="right")
    ax.set(ylabel="% of records", title="What the literature is about vs where it is written")
    ax.legend()
    fig.tight_layout()
    finish(fig, "figure2c_about_vs_from")

    pos = cmp_af[cmp_af["gap"] > 3].index.tolist()
    if pos:
        print(f"\nStudied more than authored: {', '.join(pos)}")
        print("  Research attention exceeding local authorship in a region is a")
        print("  different finding from low output, and worth reporting as such.")

---

Next: **`04_network_and_matrix.ipynb`**.